In [ ]:
%%capture
from pathlib import Path

from dj_notebook import activate

plus = activate(dotenv_file="/Users/erikvw/source/edc_source/meta-edc/.env")
report_folder = Path("/Users/erikvw/Documents/ucl/protocols/meta3/reports/")
export_folder = Path("/Users/erikvw/Documents/ucl/protocols/meta3/export/")

In [ ]:
import pandas as pd
from edc_lab_results_import.result_importer import ResultImporter
from edc_identifier.utils import is_valid_subject_identifier
from edc_lab_panel.panels import wbc_differential
from meta_labs.dataframe import get_df_bloodresults
from edc_model_to_dataframe.constants import SYSTEM_COLUMNS

In [ ]:
fname = "/tmp/results_importer_202607151936.parquet" # see imported_labs notebook to update

In [ ]:
importer = ResultImporter(
    "MNH",
    Path("~/upload/gmail").expanduser(),
    is_valid_identifier_func=is_valid_subject_identifier,
    extra_panels=[wbc_differential],
)
importer.df = pd.read_parquet(fname, engine="pyarrow")
df_results = importer.model_to_dataframe()
df_bloodresults = get_df_bloodresults()
df_requisitions = importer.df_requisitions.copy()

In [ ]:
cols = ["subject_identifier", "visit_datetime", "visit_code", "visit_code_sequence", "requisition", "subject_visit", "requisition_identifier", "panel_name", "drawn_datetime"]

df_missing = df_requisitions[cols].merge(df_bloodresults[[c for c in df_bloodresults if c not in ["panel_name"]]], on=["requisition", "subject_visit"], how="left", suffixes=["", "_y"]).query("source.isna()").copy().reset_index(drop=True)
len(df_missing)

In [ ]:
df = (
    df_missing[cols]
    .merge(df_results[[c for c in df_results.columns if c not in SYSTEM_COLUMNS]], on=["requisition", "subject_visit"], how="left", suffixes=["", "_y"])
    .query("id.isna() and panel_name != 'blood_glucose'").drop_duplicates(subset=cols, keep="first")
    .sort_values(["subject_identifier", "visit_datetime", "visit_code", "visit_code_sequence","panel_name"])
)[[c for c in cols if c not in ["requisition", "subject_visit"]]].copy().reset_index(drop=True)

df.to_csv(report_folder / "requisition_without_results_20260719.csv", index=False, date_format="%Y-%m-%d")

In [ ]:
pivoted = df.pivot_table(
    index=[
        "subject_identifier",
        "visit_datetime",
        "visit_code",
        "visit_code_sequence",
    ],
    columns="panel_name",
    values="drawn_datetime",
    aggfunc="first",
).reset_index()
pivoted.to_csv(report_folder / "requisition_without_results_pivot_20260719.csv", index=False, date_format="%Y-%m-%d")


In [ ]:
pivoted